# 08 - Association Rules by Cluster

This notebook mines simple pairwise association rules inside each final K-Means cluster. Basket data is used only after the final customer clusters are assigned.

## Scope

- Read the final `outputs/customer_clusters.csv` file only; do not modify it.
- Use `data/raw/customer_basket.csv` only for post-clustering purchasing behaviour.
- Mine pairwise association rules by cluster.
- Do not create promotion recommendations in this phase.

In [11]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../src")

from promotions import (
    add_basket_level_features,
    create_cluster_association_rules,
    join_baskets_to_clusters,
    validate_customer_clusters,
)

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_colwidth", 90)

## Load Data

Both inputs are loaded with direct `pd.read_csv` calls. The final cluster assignment is read only.

In [12]:
customer_basket = pd.read_csv("../data/raw/customer_basket.csv")
customer_clusters = pd.read_csv("../outputs/customer_clusters.csv")

print("customer_basket:", customer_basket.shape)
print("customer_clusters:", customer_clusters.shape)
customer_clusters.head()

customer_basket: (100000, 3)
customer_clusters: (33038, 2)


,customer_id,cluster
0,3,1
1,4,2
2,5,1
3,7,3
4,8,3


## Validate Final Cluster File

The association-rule phase depends on the locked final segmentation output. These checks make sure the notebook is using the expected final file.

In [13]:
validate_customer_clusters(customer_clusters)

assert customer_clusters.columns.tolist() == ["customer_id", "cluster"]
assert len(customer_clusters) == 33038
assert customer_clusters["customer_id"].duplicated().sum() == 0
assert customer_clusters["cluster"].isna().sum() == 0

print("Final cluster file validation passed.")

Final cluster file validation passed.


## Prepare Basket Records

The basket table is parsed into product lists, then joined to the final customer clusters. Customers without basket records stay in `customer_clusters`; they simply cannot contribute to basket association rules.

In [14]:
basket_features = add_basket_level_features(customer_basket)
basket_clusters = join_baskets_to_clusters(basket_features, customer_clusters)

assert basket_clusters["cluster"].notna().all()

customers_with_baskets = basket_clusters["customer_id"].nunique()
customers_without_baskets = len(customer_clusters) - customers_with_baskets

print("Basket rows with assigned cluster:", len(basket_clusters))
print("Customers with basket records:", customers_with_baskets)
print("Clustered customers without basket records:", customers_without_baskets)

basket_clusters[["invoice_id", "customer_id", "cluster", "goods_list", "basket_size"]].head()

Basket rows with assigned cluster: 100000
Customers with basket records: 28127
Clustered customers without basket records: 4911


,invoice_id,customer_id,cluster,goods_list,basket_size
0,3700630,12912,1,"[chicken, rice, pepper, whole wheat rice, shrimp, toothpaste, gums, cereals, bluetooth...",11
1,10242376,22853,1,"[low fat yogurt, tomatoes, pepper, asparagus, mint green tea, phone charger, ground be...",12
2,91550,19,0,"[cake, tomatoes, pancakes, iPad, final fantasy XIX, mineral water, ring light]",7
3,3137503,10995,4,"[cereals, megaman zero, final fantasy XIX, honey]",4
4,7165061,27807,4,"[rice, frozen smoothie, black tea, tea, minecraft, almonds, fresh bread, milk]",8


## Mine Cluster-Level Rules

The thresholds are intentionally moderate so each cluster keeps interpretable rules without creating a large experimental output. Rules are ranked by lift, then confidence, then support.

In [15]:
MIN_SUPPORT = 0.01
MIN_CONFIDENCE = 0.15
MIN_LIFT = 1.20
TOP_N_PER_CLUSTER = 20

cluster_rules = create_cluster_association_rules(
    basket_clusters,
    min_support=MIN_SUPPORT,
    min_confidence=MIN_CONFIDENCE,
    min_lift=MIN_LIFT,
    max_len=2,
    top_n=TOP_N_PER_CLUSTER,
)

cluster_rules.to_csv("../outputs/cluster_association_rules.csv", index=False)

print("Saved rows:", len(cluster_rules))
cluster_rules.head()

Saved rows: 100


,cluster,antecedents,consequents,antecedent_support,consequent_support,support,confidence,lift,leverage,conviction,rule_rank
0,0,energy drink,bluetooth headphones,0.085588,0.090311,0.019894,0.232441,2.573801,0.012165,1.185173,1
1,0,bluetooth headphones,energy drink,0.090311,0.085588,0.019894,0.220285,2.573801,0.012165,1.172753,2
2,0,energy drink,airpods,0.085588,0.155861,0.032632,0.381271,2.446226,0.019292,1.364311,3
3,0,airpods,energy drink,0.155861,0.085588,0.032632,0.209366,2.446226,0.019292,1.156557,4
4,0,bluetooth headphones,airpods,0.090311,0.155861,0.034063,0.377179,2.419973,0.019987,1.355348,5


## Rule Strength by Cluster

Each cluster is capped at the top 20 rules, so the number of displayed rules is not a measure of total association richness. Lift and confidence are more useful for comparing rule strength.

In [16]:
rule_summary = (
    cluster_rules.groupby("cluster")
    .agg(
        number_of_saved_rules=("rule_rank", "count"),
        average_lift=("lift", "mean"),
        max_lift=("lift", "max"),
        average_confidence=("confidence", "mean"),
        max_confidence=("confidence", "max"),
    )
    .reset_index()
)

rule_summary

,cluster,number_of_saved_rules,average_lift,max_lift,average_confidence,max_confidence
0,0,20,2.181905,2.573801,0.214988,0.381271
1,1,20,2.627643,2.895475,0.241857,0.382609
2,2,20,2.102798,2.448906,0.247513,0.308312
3,3,20,3.387534,3.829371,0.275355,0.462617
4,4,20,1.302880,1.392960,0.286385,0.463542


In [17]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=rule_summary, x="cluster", y="max_lift", ax=ax, color="#5DA5DA")
ax.set_title("Strongest Association Rule Lift by Cluster")
ax.set_xlabel("Cluster")
ax.set_ylabel("Max lift")
plt.tight_layout()

## Top Rules

The table below shows the strongest rules per cluster. A high lift means the consequent appears more often with the antecedent than expected from its baseline frequency.

In [18]:
top_rules = cluster_rules.sort_values(["cluster", "rule_rank"]).groupby("cluster").head(5).copy()
top_rules["rule"] = top_rules["antecedents"] + " -> " + top_rules["consequents"]

top_rules[
    ["cluster", "rule_rank", "rule", "support", "confidence", "lift"]
].round({"support": 4, "confidence": 4, "lift": 4})

,cluster,rule_rank,rule,support,confidence,lift
0,0,1,energy drink -> bluetooth headphones,0.0199,0.2324,2.5738
1,0,2,bluetooth headphones -> energy drink,0.0199,0.2203,2.5738
2,0,3,energy drink -> airpods,0.0326,0.3813,2.4462
3,0,4,airpods -> energy drink,0.0326,0.2094,2.4462
4,0,5,bluetooth headphones -> airpods,0.0341,0.3772,2.4200
20,1,1,energy drink -> bluetooth headphones,0.0208,0.2465,2.8955
21,1,2,bluetooth headphones -> energy drink,0.0208,0.2445,2.8955
22,1,3,salad -> avocado,0.0189,0.2348,2.8289
23,1,4,avocado -> salad,0.0189,0.2281,2.8289
24,1,5,spinach -> salad,0.0199,0.2199,2.7274


## Interpretation

- Cluster 0 has strong electronics and drink co-occurrence, especially `bluetooth headphones`, `airpods`, and `energy drink`.
- Cluster 1 combines a similar electronics and energy-drink signal with fresh-product pairings such as `salad` and `avocado`.
- Cluster 2 has weaker but still useful rules around `energy drink`, `airpods`, and practical grocery/home pairs like `cooking oil` and `napkins`.
- Cluster 3 has the strongest lifts, dominated by electronics accessory combinations such as `airpods`, `bluetooth headphones`, and `laptop`.
- Cluster 4 has lower lift overall and is more grocery-oriented, with everyday product pairings rather than a sharp electronics bundle.

## Caveats

- These rules describe observed baskets only; customers without basket records remain part of the final cluster output but do not contribute to rule mining.
- Association rules show co-occurrence, not causality.
- This notebook does not create recommendations yet. The next phase should translate these findings into simple cluster-level promotion candidates.

## Validation

In [19]:
saved_rules = pd.read_csv("../outputs/cluster_association_rules.csv")

expected_columns = [
    "cluster",
    "antecedents",
    "consequents",
    "antecedent_support",
    "consequent_support",
    "support",
    "confidence",
    "lift",
    "leverage",
    "conviction",
    "rule_rank",
]

assert saved_rules.columns.tolist() == expected_columns
assert saved_rules.shape == cluster_rules.shape
assert saved_rules["cluster"].nunique() == customer_clusters["cluster"].nunique()
assert Path("../outputs/cluster_association_rules.csv").exists()

print("Association-rule output validation passed.")

Association-rule output validation passed.
